In [ ]:
# ==============================
# 1. Import libraries
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# ==============================
# 2. Load dataset
# ==============================

df = pd.read_csv("/content/brca_data_w_subtypes.csv")

print(df.shape)
df.head()

(705, 1941)


,rs_CLEC3A,rs_CPB1,rs_SCGB2A2,rs_SCGB1D2,rs_TFF1,rs_MUCL1,rs_GSTM1,rs_PIP,rs_ADIPOQ,rs_ADH1B,...,pp_p62.LCK.ligand,pp_p70S6K,pp_p70S6K.pT389,pp_p90RSK,pp_p90RSK.pT359.S363,vital.status,PR.Status,ER.Status,HER2.Final.Status,histological.type
0,0.892818,6.580103,14.123672,10.606501,13.189237,6.649466,10.520335,10.338490,10.248379,10.229970,...,-0.691766,-0.337863,-0.178503,0.011638,-0.207257,0,Positive,Positive,Negative,infiltrating ductal carcinoma
1,0.000000,3.691311,17.116090,15.517231,9.867616,9.691667,8.179522,7.911723,1.289598,1.818891,...,0.279067,0.292925,-0.155242,-0.089365,0.267530,0,Positive,Negative,Negative,infiltrating ductal carcinoma
2,3.748150,4.375255,9.658123,5.326983,12.109539,11.644307,10.517330,5.114925,11.975349,11.911437,...,0.219910,0.308110,-0.190794,-0.222150,-0.198518,0,Positive,Positive,Negative,infiltrating ductal carcinoma
3,0.000000,18.235519,18.535480,14.533584,14.078992,8.913760,10.557465,13.304434,8.205059,9.211476,...,-0.266554,-0.079871,-0.463237,0.522998,-0.046902,0,Positive,Positive,Negative,infiltrating ductal carcinoma
4,0.000000,4.583724,15.711865,12.804521,8.881669,8.430028,12.964607,6.806517,4.294341,5.385714,...,-0.441542,-0.152317,0.511386,-0.096482,0.037473,0,Positive,Positive,Negative,infiltrating ductal carcinoma


ER status is biologically connected to hormone signaling, transcriptional activity, and protein-level pathway activity. Protein data may add information that RNA alone does not fully capture.

Mutation data can be useful, but for ER-status prediction it may be weaker because ER status is often more expression/pathway-driven than mutation-driven.

We compared RNA-only models with RNA-integrated multi-omics models by adding protein expression, mutation, and copy-number features to evaluate whether additional molecular layers improve ER-status prediction.

In [ ]:
# ==============================
# 3. Select features and target
# ==============================

copy_cols = [col for col in df.columns if col.startswith("cn_")]

X = df[copy_cols]
y = df["ER.Status"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (705, 860)
Target shape: (705,)


In [ ]:
# ==============================
# 4. Check target values before cleaning
# ==============================

print(y.value_counts(dropna=False))

ER.Status
Positive                       414
Negative                       135
NaN                            122
Not Performed                   27
Performed but Not Available      5
Indeterminate                    2
Name: count, dtype: int64


In [ ]:
# ==============================
# 5. Clean target labels
# Remove NaN, Indeterminate, Not Performed, etc.
# ==============================

valid_classes = ["Positive", "Negative"]

valid_rows = (
    y.notna() &
    y.isin(valid_classes)
)

X_clean = X.loc[valid_rows].copy()
y_clean = y.loc[valid_rows].copy()

print("Original rows:", len(y))
print("Cleaned rows:", len(y_clean))
print("Removed rows:", len(y) - len(y_clean))

print("\nCleaned target counts:")
print(y_clean.value_counts())

print("\nCleaned target proportions:")
print(y_clean.value_counts(normalize=True))

Original rows: 705
Cleaned rows: 549
Removed rows: 156

Cleaned target counts:
ER.Status
Positive    414
Negative    135
Name: count, dtype: int64

Cleaned target proportions:
ER.Status
Positive    0.754098
Negative    0.245902
Name: proportion, dtype: float64


In [ ]:
# ==============================
# 6. Encode labels
# ==============================

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_clean)

print("Label mapping:")
for label, code in zip(encoder.classes_, encoder.transform(encoder.classes_)):
    print(label, "=", code)

Label mapping:
Negative = 0
Positive = 1


In [ ]:
# ==============================
# 7. Train-test split
# Stratify keeps the same class ratio
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (439, 860)
X_test: (110, 860)
y_train: (439,)
y_test: (110,)


In [ ]:
# ==============================
# 8. Check class balance after split
# ==============================

print("Training class distribution:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts(normalize=True))

Training class distribution:
1    0.753986
0    0.246014
Name: proportion, dtype: float64

Testing class distribution:
1    0.754545
0    0.245455
Name: proportion, dtype: float64


In [ ]:
# ==============================
# 9. Feature selection
# IMPORTANT: fit only on training data
# ==============================

selector = SelectKBest(score_func=f_classif, k=300)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

print("Selected train shape:", X_train_selected.shape)
print("Selected test shape:", X_test_selected.shape)

Selected train shape: (439, 300)
Selected test shape: (110, 300)


In [ ]:
# ==============================
# 10. Train model
# class_weight helps with imbalance
# ==============================

model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train_selected, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score
)

# ==============================
# 1. Predictions
# ==============================

y_pred = model.predict(X_test_selected)

# predicted probabilities
y_proba = model.predict_proba(X_test_selected)[:, 1]

In [ ]:
# ==============================
# 2. Basic evaluation
# ==============================

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

Confusion Matrix:
[[18  9]
 [ 5 78]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.78      0.67      0.72        27
    Positive       0.90      0.94      0.92        83

    accuracy                           0.87       110
   macro avg       0.84      0.80      0.82       110
weighted avg       0.87      0.87      0.87       110



In [ ]:
# ==============================
# 3. Individual metrics
# ==============================

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_proba)

print("Accuracy:", accuracy)
print("Balanced Accuracy:", balanced_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

Accuracy: 0.8727272727272727
Balanced Accuracy: 0.8032128514056225
Precision: 0.896551724137931
Recall: 0.9397590361445783
F1-score: 0.9176470588235294
ROC-AUC: 0.8819723337795627


In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)
rf_model.fit(X_train_selected, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=3, min_samples_split=5,
                       n_estimators=300, random_state=42)

In [ ]:
y_pred = rf_model.predict(X_test_selected)
y_proba = rf_model.predict_proba(X_test_selected)[:, 1]
print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[20  7]
 [11 72]]
              precision    recall  f1-score   support

    Negative       0.65      0.74      0.69        27
    Positive       0.91      0.87      0.89        83

    accuracy                           0.84       110
   macro avg       0.78      0.80      0.79       110
weighted avg       0.85      0.84      0.84       110

Accuracy: 0.8363636363636363
Balanced Accuracy: 0.8041053101294064
Precision: 0.9113924050632911
Recall: 0.8674698795180723
F1-score: 0.8888888888888888
ROC-AUC: 0.8866577420794288


In [ ]:
import numpy as np

negative_count = np.sum(y_train == 0)
positive_count = np.sum(y_train == 1)

scale_pos_weight = negative_count / positive_count

print(scale_pos_weight)

0.32628398791540786


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_selected, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = xgb_model.predict(X_test_selected)
y_proba = xgb_model.predict_proba(X_test_selected)[:, 1]

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[19  8]
 [ 8 75]]
              precision    recall  f1-score   support

    Negative       0.70      0.70      0.70        27
    Positive       0.90      0.90      0.90        83

    accuracy                           0.85       110
   macro avg       0.80      0.80      0.80       110
weighted avg       0.85      0.85      0.85       110

Accuracy: 0.8545454545454545
Balanced Accuracy: 0.8036590807675146
Precision: 0.9036144578313253
Recall: 0.9036144578313253
F1-score: 0.9036144578313253
ROC-AUC: 0.8813029897367246


The selected XGBoost hyperparameters were chosen to create a balanced and stable model for a high-dimensional genomics dataset with a relatively small sample size and class imbalance.



A smaller tree depth was used to reduce overfitting and improve generalization since deeper trees can memorize noisy gene expression patterns instead of learning meaningful biological relationships.




A lower learning rate was selected to allow the model to learn gradually and more stably rather than making overly aggressive updates. The number of estimators was kept moderate to provide sufficient learning capacity while avoiding excessive model complexity.
 A higher minimum child weight was used to prevent unreliable splits formed from very small subsets of samples, which is important in genomic datasets where noise can easily influence tree construction. Subsampling was applied so that each tree learns from only a portion of the training samples, improving robustness and reducing overfitting.



 Column sampling was also used so that each tree considers only a subset of gene features, which is especially useful in high-dimensional biological data containing many potentially noisy or redundant variables.



 Additionally, scale_pos_weight was included to address class imbalance by giving greater importance to the minority class during training, helping the model avoid bias toward the majority class.

In [ ]:
from xgboost import XGBClassifier

xgb_model_changed = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.5,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

xgb_model_changed.fit(X_train_selected, y_train)
y_pred = xgb_model_changed.predict(X_test_selected)
y_proba = xgb_model_changed.predict_proba(X_test_selected)[:, 1]
print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[22  5]
 [16 67]]
              precision    recall  f1-score   support

    Negative       0.58      0.81      0.68        27
    Positive       0.93      0.81      0.86        83

    accuracy                           0.81       110
   macro avg       0.75      0.81      0.77       110
weighted avg       0.84      0.81      0.82       110

Accuracy: 0.8090909090909091
Balanced Accuracy: 0.8110218652387327
Precision: 0.9305555555555556
Recall: 0.8072289156626506
F1-score: 0.864516129032258
ROC-AUC: 0.8991521642124053


In [ ]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

The RBF kernel allows SVM to capture nonlinear relationships between genes and ER status.

The balanced class weight helps the model pay more attention to the minority class during training without changing dataset size

A moderate C value helps prevent overfitting while still allowing the model to learn meaningful class boundaries.

Gamma set to scale automatically adapts to the feature variance and is usually a safe starting point for high-dimensional biological data

In [ ]:
svm_model.fit(X_train_selected, y_train)
y_pred = svm_model.predict(X_test_selected)

y_proba = svm_model.predict_proba(X_test_selected)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[23  4]
 [ 8 75]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.74      0.85      0.79        27
    Positive       0.95      0.90      0.93        83

    accuracy                           0.89       110
   macro avg       0.85      0.88      0.86       110
weighted avg       0.90      0.89      0.89       110

Accuracy: 0.8909090909090909
Balanced Accuracy: 0.8777331548415885
Precision: 0.9493670886075949
Recall: 0.9036144578313253
F1-score: 0.9259259259259259
ROC-AUC: 0.9192324854975458


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

In [ ]:
svm_model.fit(X_train_scaled, y_train)
y_pred = svm_model.predict(X_test_scaled)

y_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[23  4]
 [10 73]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.70      0.85      0.77        27
    Positive       0.95      0.88      0.91        83

    accuracy                           0.87       110
   macro avg       0.82      0.87      0.84       110
weighted avg       0.89      0.87      0.88       110

Accuracy: 0.8727272727272727
Balanced Accuracy: 0.8656849620705043
Precision: 0.948051948051948
Recall: 0.8795180722891566
F1-score: 0.9125
ROC-AUC: 0.9174475680499777


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

In [ ]:
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42)

In [ ]:
y_pred = lr_model.predict(X_test_scaled)

y_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[19  8]
 [10 73]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.66      0.70      0.68        27
    Positive       0.90      0.88      0.89        83

    accuracy                           0.84       110
   macro avg       0.78      0.79      0.78       110
weighted avg       0.84      0.84      0.84       110

Accuracy: 0.8363636363636363
Balanced Accuracy: 0.7916108879964301
Precision: 0.9012345679012346
Recall: 0.8795180722891566
F1-score: 0.8902439024390244
ROC-AUC: 0.8688085676037484


In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    num_leaves=7,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.5,
    class_weight="balanced",
    random_state=42
)

lgbm_model.fit(X_train_selected, y_train)

y_pred = lgbm_model.predict(X_test_selected)
y_proba = lgbm_model.predict_proba(X_test_selected)[:, 1]

[LightGBM] [Info] Number of positive: 331, number of negative: 108
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000511 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1251
[LightGBM] [Info] Number of data points in the train set: 439, number of used features: 300
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[22  5]
 [13 70]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.63      0.81      0.71        27
    Positive       0.93      0.84      0.89        83

    accuracy                           0.84       110
   macro avg       0.78      0.83      0.80       110
weighted avg       0.86      0.84      0.84       110

Accuracy: 0.8363636363636363
Balanced Accuracy: 0.8290941543953592
Precision: 0.9333333333333333
Recall: 0.8433734939759037
F1-score: 0.8860759493670886
ROC-AUC: 0.892458723784025


Scientifically this is actually a strong result
**tree ensemble methods dominate linear models**
We are discovering that:


for this ER genomics problem

which is biologically plausible because gene interactions are often nonlinear.

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.05,
    depth=3,
    l2_leaf_reg=5,
    auto_class_weights="Balanced",
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=0
)

cat_model.fit(X_train_selected, y_train)

y_pred = cat_model.predict(X_test_selected)
y_proba = cat_model.predict_proba(X_test_selected)[:, 1]

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[23  4]
 [12 71]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.66      0.85      0.74        27
    Positive       0.95      0.86      0.90        83

    accuracy                           0.85       110
   macro avg       0.80      0.85      0.82       110
weighted avg       0.88      0.85      0.86       110

Accuracy: 0.8545454545454545
Balanced Accuracy: 0.8536367692994199
Precision: 0.9466666666666667
Recall: 0.8554216867469879
F1-score: 0.8987341772151899
ROC-AUC: 0.8973672467648371


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [ ]:
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def run_cv_for_model(model_name, model, X_clean, y_encoded, cv, selector):

    scaling_models = ["SVM", "Logistic Regression"]

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X_clean, y_encoded), 1):

        print("Fold:", fold)

        X_train = X_clean.iloc[train_idx]
        X_test = X_clean.iloc[test_idx]

        y_train = y_encoded[train_idx]
        y_test = y_encoded[test_idx]

        print("Train shape:", X_train.shape)
        print("Test shape:", X_test.shape)

        fold_selector = clone(selector)

        X_train_selected = fold_selector.fit_transform(X_train, y_train)
        X_test_selected = fold_selector.transform(X_test)

        selected_features = X_train.columns[fold_selector.get_support()]

        print("\nSelected Features:")
        print(selected_features.tolist())
        print(len(selected_features.tolist()))
        print(len(X_train_selected[1].tolist()))


        if model_name in scaling_models:
            scaler = StandardScaler()
            X_train_selected = scaler.fit_transform(X_train_selected)
            X_test_selected = scaler.transform(X_test_selected)

        fold_model = clone(model)

        fold_model.fit(X_train_selected, y_train)

        y_pred = fold_model.predict(X_test_selected)
        y_proba = fold_model.predict_proba(X_test_selected)[:, 1]

        fold_results.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1-score": f1_score(y_test, y_pred),
            "ROC-AUC": roc_auc_score(y_test, y_proba),
            # Weighted metrics
            "Weighted Precision": precision_score(y_test, y_pred, average="weighted"),
            "Weighted Recall": recall_score(y_test, y_pred, average="weighted"),
            "Weighted F1-score": f1_score(y_test, y_pred, average="weighted"),

            # Macro metrics
            "Macro Precision": precision_score(y_test, y_pred, average="macro"),
            "Macro Recall": recall_score(y_test, y_pred, average="macro"),
            "Macro F1-score": f1_score(y_test, y_pred, average="macro"),

            "ROC-AUC": roc_auc_score(y_test, y_proba)
})



    return pd.DataFrame(fold_results)

In [ ]:
rf_cv = run_cv_for_model("Random Forest", rf_model, X_clean, y_encoded, cv, selector)

xgb_cv = run_cv_for_model("XGBoost", xgb_model, X_clean, y_encoded, cv, selector)

lgbm_cv = run_cv_for_model("LightGBM", lgbm_model, X_clean, y_encoded, cv, selector)

cat_cv = run_cv_for_model("CatBoost", cat_model, X_clean, y_encoded, cv, selector)

svm_cv = run_cv_for_model("SVM", svm_model, X_clean, y_encoded, cv, selector)

lr_cv = run_cv_for_model("Logistic Regression", lr_model, X_clean, y_encoded, cv, selector)

Fold: 1
Train shape: (439, 860)
Test shape: (110, 860)

Selected Features:
['cn_TEKT2', 'cn_DNALI1', 'cn_RSPO1', 'cn_KIAA0754', 'cn_KCNQ4', 'cn_CITED4', 'cn_EDN2', 'cn_CLDN19', 'cn_CDC20', 'cn_ARTN', 'cn_HPDL', 'cn_TSPAN1', 'cn_CYP4B1', 'cn_CYP4Z2P', 'cn_CYP4X1', 'cn_CYP4Z1', 'cn_PDZK1IP1', 'cn_RAB3B', 'cn_DIO1', 'cn_TTC22', 'cn_PCSK9', 'cn_PRKAA2', 'cn_C1orf168', 'cn_KANK4', 'cn_FOXD3', 'cn_IL12RB2', 'cn_DIRAS3', 'cn_C1orf173', 'cn_SLC44A5', 'cn_ST6GALNAC5', 'cn_AK5', 'cn_PTGFR', 'cn_DNASE2B', 'cn_LPAR3', 'cn_CLCA2', 'cn_CLCA4', 'cn_GBP5', 'cn_TGFBR3', 'cn_BRDT', 'cn_EPHX4', 'cn_ABCA4', 'cn_COL11A1', 'cn_AMY1A', 'cn_CASQ2', 'cn_VTCN1', 'cn_SPAG17', 'cn_HSD3B2', 'cn_HMGCS2', 'cn_GZMK', 'cn_CDC20B', 'cn_CCNO', 'cn_IL6ST', 'cn_PART1', 'cn_GTF2H2B', 'cn_CARTPT', 'cn_FOXD1', 'cn_F2RL2', 'cn_PDE8B', 'cn_THBS4', 'cn_CKMT2', 'cn_HAPLN1', 'cn_EDIL3', 'cn_GPR98', 'cn_FAM81B', 'cn_C5orf27', 'cn_PCSK1', 'cn_ERAP2', 'cn_EFNA5', 'cn_TSLP', 'cn_TRIM36', 'cn_CDO1', 'cn_AQPEP', 'cn_MEGF10', 'cn_FBN2',

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Selected Features:
['cn_TEKT2', 'cn_DNALI1', 'cn_RSPO1', 'cn_KIAA0754', 'cn_KCNQ4', 'cn_CITED4', 'cn_EDN2', 'cn_CLDN19', 'cn_CDC20', 'cn_ARTN', 'cn_HPDL', 'cn_TSPAN1', 'cn_CYP4B1', 'cn_CYP4Z2P', 'cn_CYP4X1', 'cn_CYP4Z1', 'cn_PDZK1IP1', 'cn_RAB3B', 'cn_DIO1', 'cn_TTC22', 'cn_PCSK9', 'cn_PRKAA2', 'cn_C1orf168', 'cn_KANK4', 'cn_FOXD3', 'cn_IL12RB2', 'cn_DIRAS3', 'cn_C1orf173', 'cn_SLC44A5', 'cn_ST6GALNAC5', 'cn_AK5', 'cn_PTGFR', 'cn_DNASE2B', 'cn_LPAR3', 'cn_CLCA2', 'cn_CLCA4', 'cn_GBP5', 'cn_TGFBR3', 'cn_BRDT', 'cn_EPHX4', 'cn_ABCA4', 'cn_COL11A1', 'cn_AMY1A', 'cn_SPAG17', 'cn_HSD3B2', 'cn_HMGCS2', 'cn_GZMK', 'cn_CDC20B', 'cn_CCNO', 'cn_IL6ST', 'cn_PART1', 'cn_GTF2H2B', 'cn_CARTPT', 'cn_FOXD1', 'cn_F2RL2', 'cn_PDE8B', 'cn_THBS4', 'cn_CKMT2', 'cn_HAPLN1', 'cn_EDIL3', 'cn_GPR98', 'cn_FAM81B', 'cn_C5orf27', 'cn_PCSK1', 'cn_ERAP2', 'cn_EFNA5', 'cn_TSLP', 'cn_TRIM36', 'cn_CDO1', 'cn_AQPEP', 'cn_MEGF10', 'cn_FBN2', 'cn_SLC27A6', 'cn_ADAMTS19', 'cn_SHROOM1', 'cn_FSTL4', 'cn_PITX1', 'cn_C5orf20

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold: 2
Train shape: (439, 860)
Test shape: (110, 860)

Selected Features:
['cn_TEKT2', 'cn_DNALI1', 'cn_RSPO1', 'cn_KIAA0754', 'cn_KCNQ4', 'cn_CITED4', 'cn_EDN2', 'cn_CLDN19', 'cn_CDC20', 'cn_ARTN', 'cn_HPDL', 'cn_TSPAN1', 'cn_CYP4B1', 'cn_CYP4Z2P', 'cn_CYP4X1', 'cn_CYP4Z1', 'cn_PDZK1IP1', 'cn_RAB3B', 'cn_DIO1', 'cn_TTC22', 'cn_PCSK9', 'cn_PRKAA2', 'cn_C1orf168', 'cn_KANK4', 'cn_FOXD3', 'cn_IL12RB2', 'cn_DIRAS3', 'cn_C1orf173', 'cn_SLC44A5', 'cn_ST6GALNAC5', 'cn_AK5', 'cn_PTGFR', 'cn_DNASE2B', 'cn_LPAR3', 'cn_CLCA2', 'cn_CLCA4', 'cn_GBP5', 'cn_TGFBR3', 'cn_BRDT', 'cn_EPHX4', 'cn_ABCA4', 'cn_COL11A1', 'cn_AMY1A', 'cn_SPAG17', 'cn_HSD3B2', 'cn_HMGCS2', 'cn_GZMK', 'cn_CDC20B', 'cn_CCNO', 'cn_IL6ST', 'cn_PART1', 'cn_GTF2H2B', 'cn_CARTPT', 'cn_FOXD1', 'cn_F2RL2', 'cn_PDE8B', 'cn_THBS4', 'cn_CKMT2', 'cn_HAPLN1', 'cn_EDIL3', 'cn_GPR98', 'cn_FAM81B', 'cn_C5orf27', 'cn_PCSK1', 'cn_ERAP2', 'cn_EFNA5', 'cn_TSLP', 'cn_TRIM36', 'cn_CDO1', 'cn_AQPEP', 'cn_MEGF10', 'cn_FBN2', 'cn_SLC27A6', 'cn_ADAMT

In [ ]:
print(rf_cv)

In [ ]:
xgb_cv

In [ ]:
lgbm_cv

In [ ]:
cat_cv

In [ ]:
svm_cv

In [ ]:
lr_cv